In [16]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv("../.env")
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    # Automatically pulls the key from your .env file
    api_key=os.getenv("OPENROUTER_API_KEY"), 
)

In [17]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [18]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [19]:
answer = assistant.rag('How do I run Olama locally?')
print(answer)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1782086400000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}]}}, 'user_id': 'user_3FQQHH1Lt915UaNwLVXsM5EoRXu'}

In [6]:
answer = assistant.rag('How do I run Ollllama locally?')
print(answer)

I'm sorry, but the provided context does not contain information on how to run Ollllama locally.


In [7]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
)

response.output_text

'Since I am an AI, I don\'t know which specific course you are referring to! To give you the right answer, **could you please tell me the name of the course or the platform where you found it?**\n\nDepending on the type of course, here is usually how it works:\n\n**1. If it is an Online Self-Paced Course (like Coursera, Udemy, or edX):**\nYes! You can almost always join these at any time. Just click the "Enroll" or "Join" button on the website.\n\n**2. If it is a Live Cohort or University Course:**\nThese usually have specific start and end dates. If the deadline has passed, you might have to wait for the next "cohort" or semester. However, some instructors allow late entry if you email them.\n\n**3. If it is a Free Course or Newsletter:**\nYou can likely join immediately by signing up with your email.\n\n**What to do next:**\n*   **Check the "Enrollment" page:** Look for a "Join" or "Register" button.\n*   **Check the dates:** See if there is a deadline listed.\n*   **Contact the inst

In [8]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [15]:
response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
    tools=[search_tool]
)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1782086400000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}]}}, 'user_id': 'user_3FQQHH1Lt915UaNwLVXsM5EoRXu'}

In [12]:
call = response.output[0]

In [13]:
call

ResponseOutputMessage(id='msg_tmp_5de0ndo5t5n', content=[ResponseOutputText(annotations=[], text='Since I am an AI, I don\'t know which specific course you are referring to! To give you the right answer, **could you please tell me the name of the course or the platform where you found it?**\n\nDepending on the type of course, here is usually how it works:\n\n**1. If it is an Online Self-Paced Course (like Coursera, Udemy, or edX):**\nYes! You can almost always join these at any time. Just click the "Enroll" or "Join" button on the website.\n\n**2. If it is a Live Cohort or University Course:**\nThese usually have specific start and end dates. If the deadline has passed, you might have to wait for the next "cohort" or semester. However, some instructors allow late entry if you email them.\n\n**3. If it is a Free Course or Newsletter:**\nYou can likely join immediately by signing up with your email.\n\n**What to do next:**\n*   **Check the "Enrollment" page:** Look for a "Join" or "Regis

In [14]:
import json

args = json.loads(call.arguments)
args

AttributeError: 'ResponseOutputMessage' object has no attribute 'arguments'

In [13]:
call.name

'search'

In [14]:
results = search(**args)

In [15]:
result_json = json.dumps(results, indent=2)

In [16]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [17]:
messages.append(call)

In [18]:
messages.append(function_call_output)

In [19]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query": "join course registration enrollment"}', call_id='chatcmpl-tool-84c003eda72807ec', name='search', type='function_call', id='fc_tmp_d41n3e8965', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'chatcmpl-tool-84c003eda72807ec',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "bd31146b0e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "When will the course be offered next?",\n    "answer": "Summer 2027."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\

In [20]:
response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
    tools=[search_tool]
)

In [21]:
print(response.output_text)

Yes, you can still join! You can start learning whenever you want.

If you are interested in receiving a **certificate**, please keep in mind that you must submit your project while the course is still accepting submissions.

**How to get started:**
*   **Materials:** You can find the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).
*   **Workflow:** A typical workflow is to watch the lesson videos, work through the notebooks/code, and submit homework through the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/) before the listed deadlines.
*   **Registration:** You can start learning and submitting homework even without registering, as registration is primarily used to gauge interest.


In [40]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(925, 244)

In [22]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [23]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [24]:
response = openai_client.responses.create(
    model='google/gemma-4-31b-it:free',
    input=messages,
    tools=[search_tool]
)

In [25]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query": "join course enrollment registration late entry"}


In [26]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query": "join course enrollment registration late entry"}', call_id='chatcmpl-tool-b6a10bb8f20d9c93', name='search', type='function_call', id='fc_tmp_dcnze3ywg06', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'chatcmpl-tool-b6a10bb8f20d9c93',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": 

In [27]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='google/gemma-4-31b-it:free',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query": "how to join the course registration enrollment"}
iteration #2...
ASSISTANT:
Yes, you can definitely join! You can start learning and working through the materials whenever you want.

Here are a few key details to help you get started:

*   **Getting Started:** You can dive in by visiting the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).
*   **Registration:** You don't actually need a confirmation email or official registration to start; you can simply begin learning and submitting homework while the submission forms are open.
*   **Certificates:** If you are interested in receiving a certificate, please keep in mind that you must submit your project while the course is still accepting submissions.
*   **Weekly Workflow:** The typical flow is to watch t

In [28]:
def agent_loop(instructions, question, model="google/gemma-4-31b-it:free") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [29]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query": "run Ollama locally"}
iteration #2...
ASSISTANT:
To run Ollama locally, follow these steps based on your operating system:

### 1. Installation
Visit the [Ollama download page](https://ollama.com/download) and choose your OS:
*   **macOS**: Download and install the `.pkg` file.
*   **Windows**: Download and install the `.msi` file.
*   **Linux**: Run the following command in your terminal:
    ```bash
    curl -fsSL https://ollama.com/install.sh | sh
    ```

### 2. Running a Model
Once installed, you can download and run a model (e.g., Llama 3) by typing this in your terminal:
```bash
ollama run llama3
```
This command will download the model (approx. 4GB), start it locally, and open a chat-like interface for you to interact with.

### 3. Verifying the Server
To ensure the local server is running correctly, you can test the API endpoint with this command:
```bash
curl http://localhost:11434
```
You should receive a JSON response (e.g., `

'To run Ollama locally, follow these steps based on your operating system:\n\n### 1. Installation\nVisit the [Ollama download page](https://ollama.com/download) and choose your OS:\n*   **macOS**: Download and install the `.pkg` file.\n*   **Windows**: Download and install the `.msi` file.\n*   **Linux**: Run the following command in your terminal:\n    ```bash\n    curl -fsSL https://ollama.com/install.sh | sh\n    ```\n\n### 2. Running a Model\nOnce installed, you can download and run a model (e.g., Llama 3) by typing this in your terminal:\n```bash\nollama run llama3\n```\nThis command will download the model (approx. 4GB), start it locally, and open a chat-like interface for you to interact with.\n\n### 3. Verifying the Server\nTo ensure the local server is running correctly, you can test the API endpoint with this command:\n```bash\ncurl http://localhost:11434\n```\nYou should receive a JSON response (e.g., `{"models": [...]}`).\n\n### 4. Integrating with Python\nIf you want to us

In [30]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query": "Can I still join the course? enrollment deadline late registration"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! You can start whenever you want, as the videos and GitHub materials are always available.

Here are a few important details regarding your enrollment:

*   **Certificates:** If you are aiming for a certificate, please note that you must submit your project while submissions are still being accepted. Certificates are awarded to those who finish with a "live" cohort, as the process requires peer-reviewing other projects.
*   **Homework:** Completing homework is not mandatory for receiving a certificate (only the Capstone project is required), but it is highly recommended to reinforce the concepts. Points from homework contribute to your rank on the leaderboard.
*   **Registration:** You don't need a confirmation email to begin. You can start learning and submitting homework immediately, as registration is prim

'Yes, you can still join the course! You can start whenever you want, as the videos and GitHub materials are always available.\n\nHere are a few important details regarding your enrollment:\n\n*   **Certificates:** If you are aiming for a certificate, please note that you must submit your project while submissions are still being accepted. Certificates are awarded to those who finish with a "live" cohort, as the process requires peer-reviewing other projects.\n*   **Homework:** Completing homework is not mandatory for receiving a certificate (only the Capstone project is required), but it is highly recommended to reinforce the concepts. Points from homework contribute to your rank on the leaderboard.\n*   **Registration:** You don\'t need a confirmation email to begin. You can start learning and submitting homework immediately, as registration is primarily used to gauge interest.\n\n**How to get started:**\n1.  Review the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoom

In [31]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query": "join course enrollment registration"}
iteration #2...
ASSISTANT:
Yes, you can definitely join the course! You can start learning whenever you want, as the videos and GitHub materials are readily available.

Here are a few important details to keep in mind:

*   **Getting Started:** You can dive right in by visiting the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).
*   **Registration:** You don't actually need a confirmation email to start. You can begin learning and submitting homework (while the forms are open) without registering, as registration is primarily used to gauge interest.
*   **Certificates:** If you are aiming for a certificate, please note that you must submit your project while submissions are still being accepted.

'Yes, you can definitely join the course! You can start learning whenever you want, as the videos and GitHub materials are readily available.\n\nHere are a few important details to keep in mind:\n\n*   **Getting Started:** You can dive right in by visiting the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n*   **Registration:** You don\'t actually need a confirmation email to start. You can begin learning and submitting homework (while the forms are open) without registering, as registration is primarily used to gauge interest.\n*   **Certificates:** If you are aiming for a certificate, please note that you must submit your project while submissions are still being accepted. Certificates are only awarded to those who finish with a "live" cohort, as the process requires peer-revi

In [ ]:
agent_loop(instructions, "what is fifa world cup")

iteration #1...
function_call: search {"query": "what is fifa world cup"}
iteration #2...
ASSISTANT:
The FIFA World Cup is an international association football competition contested by the national teams of the members of FIFA, the governing body of association football. It is the most prestigious football tournament in the world and is held every four years.

*(Note: As a course TA for the LLM Zoomcamp, I should mention that this information is general knowledge and is not part of the course curriculum, which focuses on Large Language Models, RAG, and Vector Search.)*

Are there any other areas—perhaps related to the course materials—that you would like to explore?


'The FIFA World Cup is an international association football competition contested by the national teams of the members of FIFA, the governing body of association football. It is the most prestigious football tournament in the world and is held every four years.\n\n*(Note: As a course TA for the LLM Zoomcamp, I should mention that this information is general knowledge and is not part of the course curriculum, which focuses on Large Language Models, RAG, and Vector Search.)*\n\nAre there any other areas—perhaps related to the course materials—that you would like to explore?'

In [50]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what is fifa world cup")

iteration #1...
function_call: search {"query": "fifa world cup"}
iteration #2...
ASSISTANT:
I'm sorry, but I can only answer questions related to the course and its logistics. Since I couldn't find any information regarding the "FIFA World Cup" in the course materials, I cannot answer this question.

Are there any other course-related areas or logistics you would like to explore?


'I\'m sorry, but I can only answer questions related to the course and its logistics. Since I couldn\'t find any information regarding the "FIFA World Cup" in the course materials, I cannot answer this question.\n\nAre there any other course-related areas or logistics you would like to explore?'

In [33]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [34]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [35]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [36]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [37]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [38]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [49]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='google/gemma-4-31b-it:free')
)

In [46]:
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

True

In [54]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

print("Client created successfully")

Client created successfully


In [55]:
result = runner.loop(
    prompt='How do I run Olama locally?',
    callback=callback,
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-or-v1*************************************************************ffed. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}